In [0]:
customers = spark.table(
    "olist_project.bronze.customers"
)

from pyspark.sql.functions import col, upper, trim

silver_customers = (
    customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        trim(col("customer_city")).alias("customer_city"),
        upper(trim(col("customer_state"))).alias("customer_state")
    )
    .dropDuplicates(["customer_id"])
)

(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.customers")
)

In [0]:
orders = spark.table(
    "olist_project.bronze.orders"
)

from pyspark.sql.functions import (
    col,
    to_timestamp,
    datediff,
    when
)

silver_orders = (
    orders
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    )
)

silver_orders = (
    silver_orders
    .withColumn(
        "delivery_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delay_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")
        )
    )
    .withColumn(
        "is_late_delivery",
        when(
            col("order_delivered_customer_date")
            > col("order_estimated_delivery_date"),
            1
        ).otherwise(0)
    )
)

silver_orders = silver_orders.dropDuplicates(
    ["order_id"]
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.orders")
)

In [0]:
items = spark.table(
    "olist_project.bronze.order_items"
)

silver_items = (
    items
    .withColumn(
        "price",
        col("price").cast("double")
    )
    .withColumn(
        "freight_value",
        col("freight_value").cast("double")
    )
    .withColumn(
        "total_item_value",
        col("price") + col("freight_value")
    )
)

(
    silver_items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.order_items")
)

In [0]:
products = spark.table(
    "olist_project.bronze.products"
)

translation = spark.table(
    "olist_project.bronze.category_translation"
)

silver_products = (
    products.alias("p")
    .join(
        translation.alias("t"),
        col("p.product_category_name")
        == col("t.product_category_name"),
        "left"
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name"),
        col(
            "t.product_category_name_english"
        ).alias("category_name"),
        col("p.product_name_lenght")
            .alias("product_name_length"),
        col("p.product_description_lenght")
            .alias("product_description_length"),
        col("p.product_photos_qty"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm")
    )
)

(
    silver_products.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.products")
)



In [0]:
payments = spark.table(
    "olist_project.bronze.payments"
)

silver_payments = (
    payments
    .withColumn(
        "payment_value",
        col("payment_value").cast("double")
    )
    .withColumn(
        "payment_installments",
        col("payment_installments").cast("integer")
    )
)

(
    silver_payments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.payments")
)

In [0]:
from pyspark.sql.functions import expr

reviews = spark.table(
    "olist_project.bronze.reviews"
)

silver_reviews = (
    reviews
    .select(
        "review_id",
        "order_id",
        expr("try_cast(review_score AS INT)").alias("review_score"),
        "review_comment_title",
        "review_comment_message",
        expr("try_cast(review_creation_date AS TIMESTAMP)").alias("review_creation_date")
    )
    .dropDuplicates(["review_id"])
)

(
    silver_reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.reviews")
)

In [0]:
sellers = spark.table(
    "olist_project.bronze.sellers"
)

silver_sellers = (
    sellers
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        trim(col("seller_city")).alias("seller_city"),
        upper(trim(col("seller_state"))).alias("seller_state")
    )
    .dropDuplicates(["seller_id"])
)

silver_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "olist_project.silver.sellers"
    )